# Social Credit Scoring — Predictive Model

**Objective:** Predict an individual's Social Credit Score category (`Poor`, `Average`, `Good`) based on financial, behavioral, and civic conduct data.

**Workflow:**
1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Model Training
5. Model Evaluation
6. Feature Importance & Conclusions


## 1. Setup & Data Loading
Upload `Social_Credit_Scoring_Dataset_10000.xlsx` when prompted (or mount Google Drive).

In [ ]:
# Install/import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


In [ ]:
# Option A: Upload directly in Colab
from google.colab import files
uploaded = files.upload()  # select Social_Credit_Scoring_Dataset_10000.xlsx
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
df.head()


In [ ]:
# Option B (alternative): Load from Google Drive instead of uploading each time
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_excel('/content/drive/MyDrive/path/to/Social_Credit_Scoring_Dataset_10000.xlsx')


In [ ]:
print("Shape:", df.shape)
df.info()


In [ ]:
df.describe(include='all').T

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Check for missing values
df.isnull().sum()


In [ ]:
# Target class distribution
plt.figure()
order = ['Poor', 'Average', 'Good']
sns.countplot(x='Social_Credit_Score', data=df, order=order, palette='viridis')
plt.title('Distribution of Social Credit Score')
plt.xlabel('Social Credit Score')
plt.ylabel('Count')
plt.show()


In [ ]:
# Numeric feature distributions
numeric_cols = ['Age', 'Annual_Income', 'Traffic_Violations',
                 'Community_Service_Hours', 'Online_Fraud_Complaints']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribution of {col}')
fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()


In [ ]:
# Categorical feature counts
cat_cols = ['Employment_Status', 'Education', 'Loan_Repayment',
            'Tax_Payment', 'Utility_Bill_Payment', 'Criminal_Record']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    sns.countplot(x=col, data=df, ax=axes[i], palette='pastel')
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# Relationship between key features and target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='Social_Credit_Score', y='Annual_Income', data=df, order=order, ax=axes[0], palette='viridis')
axes[0].set_title('Annual Income vs Social Credit Score')

sns.boxplot(x='Social_Credit_Score', y='Traffic_Violations', data=df, order=order, ax=axes[1], palette='viridis')
axes[1].set_title('Traffic Violations vs Social Credit Score')
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap (numeric features only)
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()


## 3. Data Preprocessing
- Encode categorical variables
- Encode the target variable (ordinal: Poor < Average < Good)
- Split into train/test sets
- Scale numeric features


In [ ]:
df_model = df.copy()

# Binary / nominal categorical columns -> Label Encoding
binary_cat_cols = ['Loan_Repayment', 'Tax_Payment', 'Utility_Bill_Payment', 'Criminal_Record']
le_dict = {}
for col in binary_cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    le_dict[col] = le

# Multi-class nominal columns -> One-Hot Encoding
df_model = pd.get_dummies(df_model, columns=['Employment_Status', 'Education'], drop_first=True)

# Target variable -> ordinal mapping (Poor=0, Average=1, Good=2)
target_map = {'Poor': 0, 'Average': 1, 'Good': 2}
df_model['Social_Credit_Score'] = df_model['Social_Credit_Score'].map(target_map)

df_model.head()


In [ ]:
X = df_model.drop(columns=['Social_Credit_Score'])
y = df_model['Social_Credit_Score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


In [ ]:
# Scale numeric features (tree models don't need this, but useful for Logistic Regression)
scaler = StandardScaler()
num_features_to_scale = ['Age', 'Annual_Income', 'Traffic_Violations',
                          'Community_Service_Hours', 'Online_Fraud_Complaints']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_features_to_scale] = scaler.fit_transform(X_train[num_features_to_scale])
X_test_scaled[num_features_to_scale] = scaler.transform(X_test[num_features_to_scale])


## 4. Model Training
We'll train and compare three models:
- Logistic Regression (baseline, uses scaled features)
- Decision Tree
- Random Forest


In [ ]:
models = {}

# Logistic Regression (scaled data)
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
models['Logistic Regression'] = (log_reg, X_test_scaled)

# Decision Tree (raw data)
dt = DecisionTreeClassifier(max_depth=8, random_state=42)
dt.fit(X_train, y_train)
models['Decision Tree'] = (dt, X_test)

# Random Forest (raw data)
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
models['Random Forest'] = (rf, X_test)

print("Models trained: ", list(models.keys()))


## 5. Model Evaluation

In [ ]:
results = {}
for name, (model, X_te) in models.items():
    preds = model.predict(X_te)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(f"--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=['Poor', 'Average', 'Good']))
    print()


In [ ]:
# Accuracy comparison
plt.figure()
sns.barplot(x=list(results.keys()), y=list(results.values()), palette='viridis')
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')
for i, v in enumerate(results.values()):
    plt.text(i, v + 0.01, f"{v:.3f}", ha='center')
plt.show()


In [ ]:
# Confusion matrix for the best model
best_model_name = max(results, key=results.get)
best_model, X_te_best = models[best_model_name]
preds_best = best_model.predict(X_te_best)

cm = confusion_matrix(y_test, preds_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Poor', 'Average', 'Good'])
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix — {best_model_name}')
plt.show()

print(f"Best performing model: {best_model_name} (Accuracy: {results[best_model_name]:.4f})")


## 6. Feature Importance & Conclusions

In [ ]:
# Feature importance (works for tree-based models)
if best_model_name in ['Random Forest', 'Decision Tree']:
    importances = pd.Series(best_model.feature_importances_, index=X_te_best.columns)
    importances = importances.sort_values(ascending=False).head(10)

    plt.figure(figsize=(8, 6))
    sns.barplot(x=importances.values, y=importances.index, palette='viridis')
    plt.title(f'Top 10 Feature Importances — {best_model_name}')
    plt.xlabel('Importance')
    plt.show()
else:
    print("Feature importance plot only shown for tree-based models.")


### Conclusions

- Summarize which features most strongly influence the Social Credit Score (e.g., payment history, income, violations).
- Note the best-performing model and its accuracy.
- Discuss limitations: synthetic/simulated dataset, potential class imbalance, ethical considerations of social credit scoring systems.
- Suggest next steps: hyperparameter tuning (GridSearchCV), trying gradient boosting (XGBoost/LightGBM), handling class imbalance (SMOTE), deploying the model via a simple API or Streamlit app.
